In [121]:
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow import keras
from tensorflow.keras import layers
from optbinning import BinningProcess

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    log_loss,
    brier_score_loss,
)

In [122]:
SEED = 42
tf.keras.utils.set_random_seed(SEED)

In [123]:
df = pd.read_parquet('C:/Users/akshe/OneDrive/Documents/Other/Coding/Summer 2026 Project/policies_full_corrected.parquet', dtype_backend='pyarrow')

In [198]:
add_drv_num = df["driver_age"].count()
total_samples = len(df)
print(round(100*add_drv_num/total_samples, 0),"%")

36.0 %


In [124]:
df["UW_Date"] = pd.to_datetime(
    df["UW_Date"],
    origin="1899-12-30",
    unit="D"
)

In [125]:
df["claim_occurred"] = (
    df["Claim Count"].fillna(0) > 0
).astype("float32")

In [126]:
df["region"] = (
    df["ZIP_CODE"].astype(str).str[:2]
)

In [127]:
# assigning the features
all_features = [
    "veh_colour_code", 
    "lessee_gender",
    "lessee_marital", 
    "lessee_nationality", 
    "lessee_ncd_code_corrected",
    "region",
    "vehicle_type_l2", 
    "driver_gender", 
    "lessee_age", 
    "lessee_license_yrs", 
    "veh_age",
    "DEDUCTIBLE_AMOUNT",
    "has_add_drv",
    "driver_age",	
    "driver_marital", 
    "driver_license_yrs",	
    "driver_nationality",	
    "driver_relation_code", 
    "driver_ncd_code",
    "lease_year", 
    "inception_lessee_age",	
    "inception_lessee_license_yrs",	
    "inception_veh_age"
]
# Making the additional driver features separate
# The features only have values for certain samples and as such must be treated differently
add_drv_features = [
    "driver_age",	
    "driver_gender",
    "driver_marital", 
    "driver_license_yrs",	
    "driver_nationality",	
    "driver_relation_code", 
    "driver_ncd_code",
]

non_drv_features = [
]

for feature in all_features:
    if feature not in add_drv_features:
        non_drv_features.append(feature)


In [128]:
features_to_bin = [
    feature
    for feature in all_features
    if feature != "has_add_drv"
]

In [129]:
numeric_binning_features = [
    "lessee_age",
    "lessee_license_yrs",
    "veh_age",
    "DEDUCTIBLE_AMOUNT",
    "driver_age",
    "driver_license_yrs",
    "lease_year",
    "inception_lessee_age",
    "inception_lessee_license_yrs",
    "inception_veh_age",
]

In [130]:
categorical_binning_features = [
    feature
    for feature in features_to_bin
    if feature not in numeric_binning_features
]

In [131]:
print(df["driver_age"].isna().sum())
print(df["driver_age"].value_counts(dropna=False).head(10))

112959
driver_age
<NA>    112959
0.0       4414
21.0      2783
22.0      2716
25.0      2579
26.0      2555
27.0      2459
23.0      2402
28.0      2263
24.0      2225
Name: count, dtype: int64[pyarrow]


In [132]:
print(df[all_features].dtypes)

veh_colour_code                                                    int16[pyarrow]
lessee_gender                                                       int8[pyarrow]
lessee_marital                                                      int8[pyarrow]
lessee_nationality                                                 int16[pyarrow]
lessee_ncd_code_corrected                                           int8[pyarrow]
region                                                                        str
vehicle_type_l2                 dictionary<values=string, indices=int8, ordere...
driver_gender                                                      float[pyarrow]
lessee_age                                                          int8[pyarrow]
lessee_license_yrs                                                  int8[pyarrow]
veh_age                                                             int8[pyarrow]
DEDUCTIBLE_AMOUNT                                                  int16[pyarrow]
has_add_drv     

In [133]:
df["UW_Date"]=pd.to_datetime(
    df["UW_Date"],
    errors="coerce",
)

df=df.sort_values(
    "UW_Date"
).reset_index(drop=True)

In [134]:
# for feature in add_drv_features:
#     df[feature]=df[feature].astype(str)    
#     df.loc[
#         df["has_add_drv"] == 0,
#         feature,
#     ] = "[NO_ADD_DRIVER]"
# for feature in non_drv_features:
#     df[feature]=df[feature].astype(str)

df["has_add_drv"] = (
    pd.to_numeric(
        df["has_add_drv"],
        errors="coerce",
    )
    .fillna(0)
    .astype("int8")
)

for feature in numeric_binning_features:
    df[feature] = pd.to_numeric(
        df[feature],
        errors="coerce",
    )

for feature in categorical_binning_features:
    df[feature] = (
        df[feature]
        .astype("string")
        .fillna("[MISSING]")
    )

numeric_add_drv_features = [
    feature
    for feature in add_drv_features
    if feature in numeric_binning_features
]

for feature in numeric_add_drv_features:
    df.loc[
        df["has_add_drv"].eq(0),
        feature,
    ] = np.nan

categorical_add_drv_features = [
    feature
    for feature in add_drv_features
    if feature in categorical_binning_features
]

for feature in categorical_add_drv_features:
    df.loc[
        df["has_add_drv"].eq(0),
        feature,
    ] = "[NO_ADD_DRIVER]"



In [135]:
n=len(df)

train_end = int(n*0.7)
validation_end = int(n*0.85)

train_df = df.iloc[:train_end].copy()

validation_df = df.iloc[
    train_end:validation_end
].copy()

test_df = df.loc[
    validation_end:
].copy()

In [136]:
print(
    train_df["UW_Date"].min(),
    train_df["UW_Date"].max(),
)

print(
    validation_df["UW_Date"].min(),
    validation_df["UW_Date"].max(),
)

print(
    test_df["UW_Date"].min(),
    test_df["UW_Date"].max(),
)


2021-11-01 00:00:00 2024-12-09 00:00:00
2024-12-09 00:00:00 2025-05-23 00:00:00
2025-05-23 00:00:00 2025-09-30 00:00:00


In [137]:
array = train_df[all_features].to_numpy(
    dtype="str"
)

print(array.shape)

(122772, 23)


In [138]:
y_train = train_df["claim_occurred"].to_numpy(dtype="float32")
y_validation = validation_df["claim_occurred"].to_numpy(dtype="float32")
y_test = test_df["claim_occurred"].to_numpy(dtype="float32")

In [139]:
target = "claim_occurred"

# Parameters fit to each categorical feature
binning_fit_params = {
    feature: {
        # Categories below 0.2% of training records are pooled
        "cat_cutoff": 0.002,
    }
    for feature in all_features
}

# binning_process = BinningProcess(
#     variable_names=features_to_bin,
#     categorical_variables=categorical_binning_features,
    
#     # This is so no feature produces more than 6 final buckets
#     max_n_bins=6,

#     # This is to make sure each final bucket contains at least 2% of training records
#     min_bin_size=0.02,

#     # Bins are merged until the remaining bin event rates are
#     # sufficiently statistically distinguishible
#     max_pvalue=0.05,

#     binning_fit_params=binning_fit_params,

#     n_jobs=-1,
# )

binning_process = BinningProcess(
    variable_names=features_to_bin,
    categorical_variables=categorical_binning_features,

    max_n_prebins=50,
    min_prebin_size=0.01,

    max_n_bins=10,
    min_bin_size=0.01,

    # Disable this initially to determine whether
    # statistical merging is too aggressive
    max_pvalue=None,

    binning_fit_params=binning_fit_params,
    n_jobs=-1,
)

In [140]:
def prepare_binning_data(frame):
    output = frame[features_to_bin].copy()

    for feature in numeric_binning_features:
        output[feature] = pd.to_numeric(
            output[feature],
            errors="coerce",
        ).astype("float64")

    for feature in categorical_binning_features:
        output[feature] = (
            output[feature]
            .astype("string")
            .fillna("[MISSING]")
            .astype(str)
        )

    return output

In [141]:
# 2. Create raw feature data
X_train_raw = prepare_binning_data(train_df)
X_validation_raw = prepare_binning_data(validation_df)
X_test_raw = prepare_binning_data(test_df)

In [142]:
def add_gate_column(binned_frame, original_frame):
    binned_frame["has_add_drv"] = (
        pd.to_numeric(
            original_frame["has_add_drv"],
            errors="coerce",
        )
        .fillna(0)
        .astype(int)
        .astype(str)
        .to_numpy()
    )

    return binned_frame[all_features]

In [143]:
binning_process.fit(
    X_train_raw,
    y_train,
)

,variable_names,"['veh_colour_code', 'lessee_gender', ...]"
,max_n_prebins,50
,min_prebin_size,0.01
,max_n_bins,10
,min_bin_size,0.01
,categorical_variables,"['veh_colour_code', 'lessee_gender', ...]"
,binning_fit_params,"{'DEDUCTIBLE_AMOUNT': {'cat_cutoff': 0.002}, 'driver_age': {'cat_cutoff': 0.002}, 'driver_gender': {'cat_cutoff': 0.002}, 'driver_license_yrs': {'cat_cutoff': 0.002}, ...}"
,n_jobs,-1
,min_n_bins,None
,max_bin_size,None
,max_pvalue,None


In [203]:
np.unique(y_train, return_counts=True)

(array([0., 1.], dtype=float32), array([90441, 32331]))

In [ ]:
X_train_binned = pd.DataFrame(
    binning_process.transform(
        X_train_raw,    
        metric="indices",
    ),
    columns=features_to_bin,
    index=train_df.index,
)

X_validation_binned = pd.DataFrame(
    binning_process.transform(
        X_validation_raw,
        metric="indices",
    ),
    columns=features_to_bin,
    index=validation_df.index,
)

X_test_binned = pd.DataFrame(
    binning_process.transform(
        X_test_raw,
        metric="indices",
    ),
    columns=features_to_bin,
    index=test_df.index,
)

X_train_binned = add_gate_column(
    X_train_binned,
    train_df,
)

X_validation_binned = add_gate_column(
    X_validation_binned,
    validation_df,
)

X_test_binned = add_gate_column(
    X_test_binned,
    test_df,
)

In [200]:
for feature in categorical_binning_features:
    print(X_train_binned[feature].value_counts())

veh_colour_code
0     54650
6     18045
2     15280
4     12835
5      9161
8      3281
9      3137
7      2606
3      1630
1      1276
10      871
Name: count, dtype: int64
lessee_gender
0    78120
1    44652
Name: count, dtype: int64
lessee_marital
0    72651
2    48508
1     1610
3        3
Name: count, dtype: int64
lessee_nationality
0    114114
2      4897
1      2745
3      1016
Name: count, dtype: int64
lessee_ncd_code_corrected
6    60977
5    21059
2     9301
4     9207
0     6604
3     5789
1     4924
7     4895
8       16
Name: count, dtype: int64
region
9     20851
8     20849
3     16517
4     11444
6     10525
5     10070
1      9420
2      7646
7      7228
0      6777
10     1445
Name: count, dtype: int64
vehicle_type_l2
6     50956
7     22178
4     14532
5     11400
8      8353
0      4956
2      3502
3      2488
9      2205
1      1464
10      738
Name: count, dtype: int64
driver_gender
3    78356
1    23626
2    16376
0     4414
Name: count, dtype: int64
driver_marit

In [145]:
for frame in [
    X_train_binned,
    X_validation_binned,
    X_test_binned,
]:
    for feature in all_features:
        frame[feature] = frame[feature].astype(str)

In [146]:

def to_keras_inputs(frame):
    inputs = {}

    for item in all_features:
        values = (
            frame[item]
            .astype(str)
            .to_numpy(dtype="str")
            .reshape(-1, 1)
        )
        inputs[item] = tf.convert_to_tensor(
            values,
            dtype=tf.string
        )
    return inputs


In [147]:
x_train = to_keras_inputs(X_train_binned)
x_validation = to_keras_inputs(X_validation_binned)
x_test = to_keras_inputs(X_test_binned)

In [148]:
print(x_train.keys())
print(x_train["veh_colour_code"].shape)
print(x_train["driver_gender"].shape)
print(y_train.shape)

dict_keys(['veh_colour_code', 'lessee_gender', 'lessee_marital', 'lessee_nationality', 'lessee_ncd_code_corrected', 'region', 'vehicle_type_l2', 'driver_gender', 'lessee_age', 'lessee_license_yrs', 'veh_age', 'DEDUCTIBLE_AMOUNT', 'has_add_drv', 'driver_age', 'driver_marital', 'driver_license_yrs', 'driver_nationality', 'driver_relation_code', 'driver_ncd_code', 'lease_year', 'inception_lessee_age', 'inception_lessee_license_yrs', 'inception_veh_age'])
(122772, 1)
(122772, 1)
(122772,)


In [149]:
model_inputs = {}
encoded_features = []
non_drv_encoded_features = []
add_drv_encoded_features = []

for feature in all_features:
    input_layer = keras.Input(
        shape=(1,),
        dtype=tf.string,
        name=feature,
    )

    lookup_layer = layers.StringLookup(
        output_mode="int",
        mask_token=None,
        name=f"{feature}_lookup",
    )

    vocabulary_values = np.array(
        X_train_binned[feature]
        .fillna("[MISSING]")
        .astype(str)
        .tolist(),
        dtype=str,
    )
    lookup_layer.adapt(
        tf.convert_to_tensor(
            vocabulary_values,
            dtype=tf.string,
        )
    )

    category_ids = lookup_layer(input_layer)

    embedding = layers.Embedding(
        input_dim=lookup_layer.vocabulary_size(),
        output_dim=4,
        name=f"{feature}_embedding",
    )(category_ids)

    embedding_vector = layers.Flatten()(embedding)

    model_inputs[feature] = input_layer

    encoded_features.append(embedding_vector)

    if feature in add_drv_features:
        add_drv_encoded_features.append(
            embedding_vector
        )
    else:
        non_drv_encoded_features.append(
            embedding_vector
        )

In [150]:
print(lookup_layer.get_vocabulary()[:20])
print(lookup_layer.vocabulary_size())

['[UNK]', np.str_('0'), np.str_('1'), np.str_('2')]
4


In [151]:
# numeric_input = keras.Input(
#     shape=(len(numeric_features),),
#     dtype=tf.float32,
#     name="numeric_features",
# )

# #This is for numeric features, there currently are none

In [152]:
# # Normalising the numeric features

# # First creating the layer
# numeric_normalizer = layers.Normalization(
#     name="numeric_normalization"
# )
# # Then the training means and variances
# numeric_normalizer.adapt(
#     train_df[numeric_features]
#     .to_numpy(dtype="float32")
# )

# normalized_numeric = numeric_normalizer(
#     numeric_input
# )

In [153]:
# model_inputs["numeric_features"] = numeric_input
# encoded_features.append(normalized_numeric)

In [154]:
add_drv_vector = layers.Concatenate(
    name="additional_driver_vector"
)(
    add_drv_encoded_features
)

non_drv_vector = layers.Concatenate(
    name="non_additional_driver_vector"
)(
    non_drv_encoded_features
)

In [155]:
print(train_df["has_add_drv"].unique())
print(train_df["has_add_drv"].dtype)

[0 1]
int8


In [156]:
has_add_drv_input = model_inputs["has_add_drv"]

# Convert:
# "1" -> 1.0
# "0" -> 0.0
# Only this separate gating branch becomes numeric
has_add_drv_gate = keras.ops.cast(
    keras.ops.equal(has_add_drv_input, "1"),
    dtype="float32",
)
has_add_drv_gate_expanded = keras.ops.repeat(
    has_add_drv_gate,
    repeats=int(add_drv_vector.shape[-1]),
    axis=1,
)

gated_add_drv_vector = layers.Multiply(
    name="gate_additional_driver_features",
)([
    add_drv_vector,
    has_add_drv_gate_expanded,
])

In [157]:
combined_features = layers.Concatenate(
    name="combined_features"
)([
    non_drv_vector,
    has_add_drv_gate,
    gated_add_drv_vector,
])

In [158]:
# This connects the input values to each neuron
# ReLU is the activation function
x = layers.Dense(
    units=64,
    activation="relu",
    name="dense_1",
)(combined_features)

In [159]:
# This allows for the network to not be too dependent on specific neurons
# This is done by temporarily setting 20% of the hidden outputs to zero
x = layers.Dropout(
    rate=0.20,
    name="dropout_1",
)(x)

In [160]:
x = layers.Dense(
    units=32,
    activation="relu",
    name="dense_2",
)(x)

In [161]:
# This is the output layer
# The sigmoid function is used to calculate the final probability
claim_probability = layers.Dense(
    units=1,
    activation="sigmoid",
    name="claim_probability",
)(x)

In [162]:
for k, v in model_inputs.items():
    print(k, type(v), getattr(v, "name", None))

veh_colour_code <class 'keras.src.backend.common.keras_tensor.KerasTensor'> veh_colour_code
lessee_gender <class 'keras.src.backend.common.keras_tensor.KerasTensor'> lessee_gender
lessee_marital <class 'keras.src.backend.common.keras_tensor.KerasTensor'> lessee_marital
lessee_nationality <class 'keras.src.backend.common.keras_tensor.KerasTensor'> lessee_nationality
lessee_ncd_code_corrected <class 'keras.src.backend.common.keras_tensor.KerasTensor'> lessee_ncd_code_corrected
region <class 'keras.src.backend.common.keras_tensor.KerasTensor'> region
vehicle_type_l2 <class 'keras.src.backend.common.keras_tensor.KerasTensor'> vehicle_type_l2
driver_gender <class 'keras.src.backend.common.keras_tensor.KerasTensor'> driver_gender
lessee_age <class 'keras.src.backend.common.keras_tensor.KerasTensor'> lessee_age
lessee_license_yrs <class 'keras.src.backend.common.keras_tensor.KerasTensor'> lessee_license_yrs
veh_age <class 'keras.src.backend.common.keras_tensor.KerasTensor'> veh_age
DEDUCTIBLE

In [163]:
# This conects the defined inputs, transformations and output into one trainable object
model = keras.Model(
    inputs=model_inputs,
    outputs=claim_probability,
    name="claim_occurrence_embedding_model",
)

In [164]:
# This is to check that the model is running as it should
model.summary()

Model: "claim_occurrence_embedding_model"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ driver_gender       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ driver_age          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ driver_marital      │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ driver_license_yrs  │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ driver_nationality  │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ driver_relation_co… │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ driver_ncd_code     │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ veh_colour_code     │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lessee_gender       │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lessee_marital      │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lessee_nationality  │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lessee_ncd_code_co… │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ region (InputLayer) │ (None, 1)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ vehicle_type_l2     │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lessee_age          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lessee_license_yrs  │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ veh_age             │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                 

 Total params: 8,829 (34.49 KB)

 Trainable params: 8,829 (34.49 KB)

 Non-trainable params: 0 (0.00 B)

In [165]:
model.compile(
    # This sets up how the model will respond to data from different batches
    optimizer=keras.optimizers.Adam( 
        learning_rate=0.001
    ),
    # This is the training loss for a binary outcome
    # As this model is only for wether there are claims or not, there are two possible outcomes
    loss=keras.losses.BinaryCrossentropy(),
    metrics=[
        keras.metrics.AUC(
            name="roc_auc",
            curve="ROC",
        ),
        keras.metrics.AUC(
            name="pr_auc",
            curve="PR",
        ),
    ],
)

In [166]:
# 
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss", # This looks at the loss on the validation dataset after each epoch (calculated from binary cross entropy)
    mode="min", # Lower is better
    patience=4, # This stops the training once there is no imporvement for 4 epochs in a row
    restore_best_weights=True, # This returns it to the version of the model that performed best
)

In [167]:
print(x_train["has_add_drv"].dtype)
print(x_train["has_add_drv"][:5])

print(has_add_drv_input.dtype)
print(has_add_drv_gate.dtype)
print(has_add_drv_gate_expanded.dtype)
print(has_add_drv_gate_expanded.shape)

<dtype: 'string'>
tf.Tensor(
[[b'0']
 [b'1']
 [b'1']
 [b'0']
 [b'1']], shape=(5, 1), dtype=string)
string
float32
float32
(None, 28)


In [168]:
history = model.fit(
    x=x_train,
    y=y_train,
    validation_data=(
        x_validation,
        y_validation,
    ),
    epochs=50, # This sets the maximum number of passes as 50. It will likely stop earlier because of early_stopping
    batch_size=2048, # The model processes 2048 policies, calculates the average loss, and updates its weights accordingly
    callbacks=[early_stopping],
    verbose=2,
)

Epoch 1/50
60/60 - 10s - 164ms/step - loss: 0.5847 - pr_auc: 0.2812 - roc_auc: 0.5424 - val_loss: 0.5757 - val_pr_auc: 0.3335 - val_roc_auc: 0.5817
Epoch 2/50
60/60 - 2s - 25ms/step - loss: 0.5612 - pr_auc: 0.3543 - roc_auc: 0.6119 - val_loss: 0.5722 - val_pr_auc: 0.3437 - val_roc_auc: 0.5965
Epoch 3/50
60/60 - 2s - 27ms/step - loss: 0.5586 - pr_auc: 0.3620 - roc_auc: 0.6206 - val_loss: 0.5712 - val_pr_auc: 0.3463 - val_roc_auc: 0.5988
Epoch 4/50
60/60 - 2s - 26ms/step - loss: 0.5578 - pr_auc: 0.3644 - roc_auc: 0.6228 - val_loss: 0.5708 - val_pr_auc: 0.3477 - val_roc_auc: 0.5998
Epoch 5/50
60/60 - 1s - 24ms/step - loss: 0.5574 - pr_auc: 0.3657 - roc_auc: 0.6242 - val_loss: 0.5705 - val_pr_auc: 0.3484 - val_roc_auc: 0.6006
Epoch 6/50
60/60 - 1s - 25ms/step - loss: 0.5571 - pr_auc: 0.3677 - roc_auc: 0.6250 - val_loss: 0.5705 - val_pr_auc: 0.3482 - val_roc_auc: 0.6006
Epoch 7/50
60/60 - 1s - 25ms/step - loss: 0.5567 - pr_auc: 0.3687 - roc_auc: 0.6257 - val_loss: 0.5704 - val_pr_auc: 0.348

In [169]:
print(model_inputs.keys())

dict_keys(['veh_colour_code', 'lessee_gender', 'lessee_marital', 'lessee_nationality', 'lessee_ncd_code_corrected', 'region', 'vehicle_type_l2', 'driver_gender', 'lessee_age', 'lessee_license_yrs', 'veh_age', 'DEDUCTIBLE_AMOUNT', 'has_add_drv', 'driver_age', 'driver_marital', 'driver_license_yrs', 'driver_nationality', 'driver_relation_code', 'driver_ncd_code', 'lease_year', 'inception_lessee_age', 'inception_lessee_license_yrs', 'inception_veh_age'])


In [170]:
test_predictions = model.predict(
    x_test,
    batch_size=4096
).reshape(-1)

7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 124ms/step


In [171]:
print("Prediction minimum:", test_predictions.min())
print("Prediction maximum:", test_predictions.max())
print("Prediction standard deviation:", test_predictions.std())
print(
    "Number of distinct rounded predictions:",
    np.unique(test_predictions.round(6)).size,
)

Prediction minimum: 0.054961864
Prediction maximum: 0.5527243
Prediction standard deviation: 0.07539843
Number of distinct rounded predictions: 24024


In [172]:
binning_summary = binning_process.summary()

print(
    binning_summary[
        [
            "name",
            "dtype",
            "status",
            "n_bins",
            "quality_score",
        ]
    ]
)

                            name        dtype   status n_bins quality_score
0                veh_colour_code  categorical  OPTIMAL     11      0.001989
1                  lessee_gender  categorical  OPTIMAL      2      0.004606
2                 lessee_marital  categorical  OPTIMAL      4      0.072148
3             lessee_nationality  categorical  OPTIMAL      4      0.004045
4      lessee_ncd_code_corrected  categorical  OPTIMAL      9      0.014825
5                         region  categorical  OPTIMAL     11      0.336638
6                vehicle_type_l2  categorical  OPTIMAL     11      0.014877
7                  driver_gender  categorical  OPTIMAL      4      0.003522
8                     lessee_age    numerical  OPTIMAL     10      0.078986
9             lessee_license_yrs    numerical  OPTIMAL     10      0.006744
10                       veh_age    numerical  OPTIMAL      4      0.053611
11             DEDUCTIBLE_AMOUNT    numerical  OPTIMAL      5      0.025616
12          

In [173]:
feature_binner = binning_process.get_binned_variable(
    "region"
)

region_table = feature_binner.binning_table.build()

print(region_table)

                                                      Bin   Count  Count (%)  \
0                            [53, 48, 28, 74, 63, 87, 43]    6777   0.055200   
1                [54, 86, 65, 77, 57, 68, 89, 25, 67, 83]    9420   0.076728   
2                                [46, 85, 64, 15, 17, 39]    7646   0.062278   
3                [52, 84, 82, 29, 66, 58, 42, 76, 56, 55]   16517   0.134534   
4                                        [18, 62, 35, 72]   11444   0.093213   
5                                    [73, 26, 16, 47, 19]   10070   0.082022   
6                                            [24, 22, 36]   10525   0.085728   
7                                                    [23]    7228   0.058873   
8                                            [32, 34, 13]   20849   0.169819   
9                                                [12, 14]   20851   0.169835   
10      [37, 44, 69, 27, 33, 75, 49, 45, 88, 11, 61, 2...    1445   0.011770   
11                                      

In [174]:
# Checking that all values are between 0 and 1
print(test_predictions.min())
print(test_predictions.max())

0.054961864
0.5527243


In [175]:
roc_auc = roc_auc_score(
    y_test,
    test_predictions,
)

pr_auc = average_precision_score(
    y_test,
    test_predictions,
)

test_log_loss = log_loss(
    y_test,
    test_predictions,
)

brier = brier_score_loss(
    y_test,
    test_predictions,
)

In [176]:
print("ROC AUC:", roc_auc)
print("PR AUC:", pr_auc)
print("Log loss:", test_log_loss)
print("Brier score:", brier)

ROC AUC: 0.6101641848336544
PR AUC: 0.2844666652132893
Log loss: 0.508469820022583
Brier score: 0.1646578460931778


In [177]:
actual_claim_rate = y_test.mean()
predicted_claim_rate = test_predictions.mean()

print("Actual claim rate:", actual_claim_rate)
print("Predicted claim rate:", predicted_claim_rate)

Actual claim rate: 0.21281691
Predicted claim rate: 0.24674895


In [178]:
results = pd.DataFrame({
    "actual": y_test,
    "predicted": test_predictions,
})

results["risk_decile"] = pd.qcut(
    results["predicted"],
    q=10,
    labels=False,
    duplicates="drop",
)

decile_table = (
    results
    .groupby("risk_decile")
    .agg(
        policies=("actual", "size"),
        actual_claim_rate=("actual", "mean"),
        predicted_claim_rate=("predicted", "mean"),
    )
)

print(decile_table)

             policies  actual_claim_rate  predicted_claim_rate
risk_decile                                                   
0                2631           0.109844              0.128873
1                2631           0.141011              0.169026
2                2631           0.163816              0.192765
3                2631           0.182820              0.212675
4                2631           0.204485              0.231596
5                2630           0.215589              0.250222
6                2631           0.228430              0.270453
7                2631           0.270620              0.294635
8                2631           0.282402              0.325341
9                2631           0.329152              0.391904


In [179]:
model.save(
    "claim_occurrence_embedding_model.keras"
)

loaded_model = keras.models.load_model(
    "claim_occurrence_embedding_model.keras"
)

In [180]:
loaded_predictions = loaded_model.predict(
    x_test,
    batch_size=4096,
).reshape(-1)

print(
    np.max(
        np.abs(
            loaded_predictions
            - test_predictions
        )
    )
)

7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 133ms/step
0.0
